In [28]:
using LowLevelFEM

# Axisymmetric Linear Elasticity

This example demonstrates the solution of a two-dimensional axisymmetric
linear elasticity problem using LowLevelFEM.

The problem is solved in two different ways:

1. using the built-in axisymmetric elasticity solver,
2. using the weak-form DSL.

The two solutions are then compared to verify the correctness of the
DSL implementation.

In [29]:
structured_rect_mesh(x0=20, lx=80, ly=20, n=80, order=2)
mat = Material("body");

## Geometry and material

A quadratic structured mesh is generated for a hollow cylinder.
The inner radius avoids the singularity associated with the axis of
revolution.

Linear isotropic elastic material properties are assigned to the body.

## Built-in axisymmetric formulation

First, the problem is solved using the dedicated axisymmetric elasticity
implementation available in LowLevelFEM.

The right boundary is subjected to a radial traction, while the lower-left
corner is constrained to remove rigid body motion.

In [30]:
prob = Problem([mat], type=:AxiSymmetric)

ld = load("left", fx=1);

In [31]:
bc = BoundaryCondition("leftbottom", uy=0);

In [32]:
u1 = solveDisplacement(prob, load=[ld], support=[bc]);

In [33]:
@time K1 = stiffnessMatrix(prob)
K1[:, :]

  1.597073 seconds (11.39 M allocations: 2.756 GiB, 28.10% gc time)


206402×206402 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6565404 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠻⠿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠒⠦⢤⣀⡀⠀⎥
⎢⣿⡆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⎥
⎢⣿⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⢳⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [34]:
S1 = solveStress(u1);

In [35]:
showDoFResults(u1, name="u1", visible=true, factor=1e5)
showStressResults(S1, name="σ1 - von Mises");

## Reference solution

The displacement and von Mises stress obtained from the built-in
axisymmetric formulation will be used as the reference solution.

## Weak-form DSL formulation

The same problem is assembled using the LowLevelFEM weak-form DSL.

The axisymmetric strain-displacement operator is written directly from
its mathematical definition.

In [36]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u);

In [37]:
r = ScalarField(Pu, "body", (x, y, z)->x);

### Axisymmetric strain operator

The strain vector is written as

$$
\varepsilon =
\begin{bmatrix}
\partial u_r/\partial r \\
u_r/r \\
\partial u_z/\partial z \\
\partial u_r/\partial z + \partial u_z/\partial r
\end{bmatrix},
$$

which is represented as

$$
\varepsilon = A_1 \nabla^s u + A_2 u.
$$

This decomposition allows the operator to be expressed naturally in the DSL.

In [38]:
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]

4×3 Matrix{Int64}:
 1  0  0
 0  0  0
 0  1  0
 0  0  1

In [39]:
A2 = [0 0; 1/r 0; 0 0; 0 0];

In [40]:
B = full(A1 ⋅ SymGrad(Pu)) + reduced(A2 ⋅ Pu);

In [41]:
E = mat.E
ν = mat.ν

D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

### System assembly

The global stiffness matrix is assembled from the weak form

$$
K = \int_\Omega B^\mathrm{T} D B \; 2\pi r \; d\Omega.
$$

The factor $2\pi r$ accounts for the axisymmetric volume measure.

In [42]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r))
K2[:, :]

  5.870349 seconds (34.83 M allocations: 3.756 GiB, 17.79% gc time)


206402×206402 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6566056 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠻⠿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠒⠦⢤⣀⡀⠀⎥
⎢⣿⡆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⎥
⎢⣿⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⢳⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [43]:
f2 = ∫(Pu ⋅ [1, 0] * (2π*r), Γ="left");

In [44]:
using Profile

# Compile and initialize all relevant methods.
K2_warmup = ∫(B' ⋅ D ⋅ B * (2π * r))

GC.gc()
Profile.clear()

@time begin
    @profile K2_profiled = ∫(B' ⋅ D ⋅ B * (2π * r))
end

Profile.print(
    format=:flat,
    sortedby=:count,
    mincount=10,
    C=false
)

  7.460065 seconds (34.83 M allocations: 3.756 GiB, 12.55% gc time)
 Count  Overhead File                    Line Function
 =====  ======== ====                    ==== ========
    10         0 @LowLevelFEM]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl\/src/multifield.jl]8;;\ 1185 detect_pattern!(pattern::LowLevel…
    10        10 @Base/]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.6%2B0.x64.linux.gnu/share/julia/base/essentials.jl\essentials.jl]8;;\     799 ifelse
    10         0 @Base/]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.6%2B0.x64.linux.gnu/share/julia/base/abstractarray.jl\abstractarray.jl]8;;\  725 checkbounds_indices
    10         0 @LowLevelFEM]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl\/src/multifield.jl]8;;\ 1193 sparse_enough
    10         0 @LowLevelFEM]8;;file://perebal-Latitude-5580/home

In [45]:
Profile.print(
    format=:tree,
    sortedby=:count,
    mincount=10,
    C=false
)

Overhead ╎ [+additional indent] Count File:Line  Function
  10╎10    ]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.6%2B0.x64.linux.gnu/share/julia/base/abstractarray.jl\@Base]8;;\]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.6%2B0.x64.linux.gnu/share/julia/base/abstractarray.jl\/abstractarray.jl:1079]8;;\  copyto_unaliased!(deststyle::IndexLinea…
  12╎12    ]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.6%2B0.x64.linux.gnu/share/julia/base/abstractarray.jl\@Base]8;;\]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.6%2B0.x64.linux.gnu/share/julia/base/abstractarray.jl\/abstractarray.jl:1063]8;;\  copyto!(dest::SubArray{Float64, 2, Matr…
  14╎14    ]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl\@LowLevelFEM]8;;\]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl\/…field.

## Solution

The linear system assembled from the DSL is solved using the same boundary
conditions as the reference formulation.

In [46]:
u2 = solveField(K2, f2, support=[bc]);

In [47]:
showDoFResults(u2, name="u2", visible=true, factor=1e5);

## Stress recovery

The strain components are computed from the displacement field,
followed by Hooke's law to obtain the stress tensor.

Finally, the von Mises equivalent stress is evaluated.

In [48]:
εr = ∂x(u2[1])
εφ = u2[1] / r
εz = ∂y(u2[2])
γrz = ∂y(u2[1]) + ∂x(u2[2]);

In [49]:
ε = [εr, εφ, εz, γrz];

In [50]:
σ = D * ε;

In [51]:
σr = σ[1]
σφ = σ[2]
σz = σ[3]
τrz = σ[4];

In [52]:
σeqv2 = √(((σr - σφ)^2 + (σφ - σz)^2 + (σz - σr)^2) / 2 + 3τrz^2);

In [53]:
showElementResults(σeqv2, name="σ2 - von Mises");

## Verification

The displacement and stress fields obtained from the weak-form DSL agree
with those produced by the built-in axisymmetric formulation, confirming
the correctness of the DSL implementation.

In [54]:
openPostProcessor()